# D26: Training Effectiveness Data Analysis
## Notebook 01: Data Ingestion, Quality Auditing & Cleaning Pipeline
**Academic Institution:** ABES Engineering College | MCA 2nd Year  
**Prepared For:** Program Manager, EdTech, HCLTech | HR Leadership  
**Domain:** Human Resources Analytics  
**Sprint Focus:** Sprint 1 (`v0.1.0`) - Data cleaning, outlier treatment, and feature engineering  

---
### Notebook Objectives:
1. Ingest raw employee training dataset (`data/raw/training_effectiveness_data_analysis.csv`).
2. Audit data quality: missing values, duplicates, invalid dates, category inconsistencies, and extreme outliers.
3. Apply structured, reproducible cleaning steps.
4. Engineer analytical KPI features (`rating_improvement`, `rating_uplift_pct`, `training_tier`, `is_flight_risk`).
5. Export clean dataset to `data/clean/training_effectiveness_cleaned.csv`.


In [1]:
# Import requisite libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
pd.set_option('display.max_columns', None)


### 1. Ingest Raw Dataset & Baseline Quality Audit

In [2]:
raw_data_path = os.path.join("..", "data", "raw", "training_effectiveness_data_analysis.csv")
# Fallback if running from root
if not os.path.exists(raw_data_path):
    raw_data_path = os.path.join("data", "raw", "training_effectiveness_data_analysis.csv")

df_raw = pd.read_csv(raw_data_path)
print(f"Loaded Raw Dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head()


Loaded Raw Dataset: 5050 rows, 22 columns


,employee_id,employee_name,department,job_role,gender,city,state,join_date,exit_date,experience_years,salary,pre_training_rating,training_program_name,training_type,training_hours,training_score,performance_rating,attendance_days,late_arrivals,manager_id,hr_cost,attrition_flag
0,EMP-1001,Daksh Bakshi,Customer Success,Technical Support Lead,Male,Noida,Uttar Pradesh,2024-01-21,2024-06-17,3.5,922000.0,3.2,Advanced Data Analytics & Power BI,Technical,46.1,73.3,3.9,220,3.0,MGR-1108,47066,Yes
1,EMP-1002,Laksh Konda,Engineering,QA Engineer,Male,Pune,Maharashtra,2024-07-08,NaN,4.6,1296000.0,3.0,Full-Stack Cloud & DevOps,Technical,7.4,40.6,2.6,226,12.0,MGR-1044,10893,No
2,EMP-1003,Isaac Dewan,Operations,Operations Manager,Male,Bengaluru,Karnataka,2025-01-22,NaN,4.5,933000.0,3.2,Enterprise Consultative Selling,Domain,9.5,46.4,2.9,204,10.0,MGR-1060,11161,No
3,EMP-1004,Charan Chahal,Finance,Financial Analyst,Male,Bengaluru,Karnataka,2025-06-04,NaN,2.6,860000.0,3.4,Strategic Leadership & Agile Management,Leadership,41.3,69.6,4.2,236,3.0,MGR-1023,51028,No
4,EMP-1005,Udyati Balay,Sales & Marketing,Digital Marketing Specialist,Female,Gurugram,Haryana,2025-09-04,NaN,2.1,803000.0,3.0,Strategic Leadership & Agile Management,Leadership,30.2,77.4,4.1,227,7.0,MGR-1015,38713,No


In [3]:
# Inspect data types and null counts
df_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5050 entries, 0 to 5049
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   employee_id            5050 non-null   object 
 1   employee_name          5050 non-null   object 
 2   department             5050 non-null   object 
 3   job_role               5050 non-null   object 
 4   gender                 5050 non-null   object 
 5   city                   5050 non-null   object 
 6   state                  5050 non-null   object 
 7   join_date              5050 non-null   object 
 8   exit_date              656 non-null    object 
 9   experience_years       5050 non-null   float64
 10  salary                 4849 non-null   float64
 11  pre_training_rating    5050 non-null   float64
 12  training_program_name  5050 non-null   object 
 13  training_type          5050 non-null   object 
 14  training_hours         4860 non-null   float64
 15  trai

In [4]:
# Identify missing value percentages across fields
null_pct = (df_raw.isnull().sum() / len(df_raw)) * 100
null_summary = pd.DataFrame({'Missing Count': df_raw.isnull().sum(), 'Missing %': null_pct.round(2)})
null_summary[null_summary['Missing Count'] > 0]


,Missing Count,Missing %
exit_date,4394,87.01
salary,201,3.98
training_hours,190,3.76
late_arrivals,212,4.20


### 2. Deduplication Audit & Resolution

In [5]:
duplicates_count = df_raw.duplicated().sum()
print(f"Total duplicate rows detected: {duplicates_count} ({duplicates_count/len(df_raw)*100:.2f}%)")

# Drop duplicate entries
df_cleaned = df_raw.drop_duplicates().reset_index(drop=True)
print(f"Post-deduplication records: {len(df_cleaned)}")


Total duplicate rows detected: 50 (0.99%)
Post-deduplication records: 5000


### 3. Rectification of Invalid Date Formats

In [6]:
# Identify anomalies in date formats
print("Known invalid dates being corrected:")
print(" - '2025-02-31' -> '2025-02-28'")
print(" - '2024-13-45' -> '2024-12-15'")

df_cleaned['join_date'] = df_cleaned['join_date'].replace({
    '2025-02-31': '2025-02-28',
    '2024-13-45': '2024-12-15'
})

df_cleaned['join_date'] = pd.to_datetime(df_cleaned['join_date'], errors='coerce')
df_cleaned['exit_date'] = pd.to_datetime(df_cleaned['exit_date'], errors='coerce')

# Verify date range
print(f"Join Date Min: {df_cleaned['join_date'].min()}, Max: {df_cleaned['join_date'].max()}")


Known invalid dates being corrected:
 - '2025-02-31' -> '2025-02-28'
 - '2024-13-45' -> '2024-12-15'
Join Date Min: 2024-01-01 00:00:00, Max: 2025-12-30 00:00:00


### 4. Categorical Normalization: Department Casing & Synonyms

In [7]:
print("Raw Department categories count:", len(df_cleaned['department'].unique()))
print(df_cleaned['department'].value_counts().head(10))

def standardize_department(val):
    if not isinstance(val, str):
        return "Engineering"
    val_clean = val.strip().lower()
    if "tech" in val_clean or "it" in val_clean or "engine" in val_clean:
        return "Engineering"
    elif "data" in val_clean or "ai" in val_clean or "science" in val_clean:
        return "Data Science & AI"
    elif "sale" in val_clean or "market" in val_clean:
        return "Sales & Marketing"
    elif "hr" in val_clean or "human" in val_clean or "people" in val_clean:
        return "Human Resources"
    elif "finan" in val_clean or "account" in val_clean:
        return "Finance"
    elif "operat" in val_clean or "process" in val_clean or "supply" in val_clean:
        return "Operations"
    elif "cust" in val_clean or "supp" in val_clean or "succ" in val_clean:
        return "Customer Success"
    return "Engineering"

df_cleaned['department'] = df_cleaned['department'].apply(standardize_department)
print('\nCleaned Department distribution:')
df_cleaned['department'].value_counts()


Raw Department categories count: 24
department
Engineering          1343
Sales & Marketing     872
Data Science & AI     772
Finance               504
Operations            491
Human Resources       483
Customer Success      416
engineering            18
IT / Tech              17
ENGINEERING            11
Name: count, dtype: int64

Cleaned Department distribution:


department
Engineering          1389
Sales & Marketing     891
Data Science & AI     790
Finance               515
Operations            500
Human Resources       495
Customer Success      420
Name: count, dtype: int64

### 5. Outlier Detection & Treatment

In [8]:
# Detect Salary and Training Hours outliers
print("Salary Min/Max before cleaning:", df_cleaned['salary'].min(), df_cleaned['salary'].max())
print("Training Hours Min/Max before cleaning:", df_cleaned['training_hours'].min(), df_cleaned['training_hours'].max())

# Flag and set outliers to NaN for domain-aligned median imputation
invalid_salary = (df_cleaned['salary'] < 300000) | (df_cleaned['salary'] > 4500000)
invalid_hours = (df_cleaned['training_hours'] < 0) | (df_cleaned['training_hours'] > 80)

print(f"Flagged {invalid_salary.sum()} salary outliers and {invalid_hours.sum()} training hours outliers.")
df_cleaned.loc[invalid_salary, 'salary'] = np.nan
df_cleaned.loc[invalid_hours, 'training_hours'] = np.nan


Salary Min/Max before cleaning: -45000.0 99999999.0
Training Hours Min/Max before cleaning: -15.0 999.0
Flagged 2 salary outliers and 2 training hours outliers.


### 6. Missing Value Imputation using Domain Group Medians

In [9]:
# Impute Salary by job role median
role_sal_median = df_cleaned.groupby('job_role')['salary'].transform('median')
dept_sal_median = df_cleaned.groupby('department')['salary'].transform('median')
df_cleaned['salary'] = df_cleaned['salary'].fillna(role_sal_median).fillna(dept_sal_median).astype(int)

# Impute Training Hours by program median
prog_hours_median = df_cleaned.groupby('training_program_name')['training_hours'].transform('median')
df_cleaned['training_hours'] = df_cleaned['training_hours'].fillna(prog_hours_median).round(1)

# Impute Late Arrivals by department median
dept_late_median = df_cleaned.groupby('department')['late_arrivals'].transform('median')
df_cleaned['late_arrivals'] = df_cleaned['late_arrivals'].fillna(dept_late_median).round().astype(int)

# Verify no remaining missing values except exit_date
df_cleaned[['salary', 'training_hours', 'late_arrivals']].isnull().sum()


salary            0
training_hours    0
late_arrivals     0
dtype: int64

### 7. Feature Engineering for Analytics & KPI Evaluation

In [10]:
# 1. Performance Uplift (Absolute & Percentage)
df_cleaned['rating_improvement'] = (df_cleaned['performance_rating'] - df_cleaned['pre_training_rating']).round(2)
df_cleaned['rating_uplift_pct'] = ((df_cleaned['rating_improvement'] / df_cleaned['pre_training_rating']) * 100).round(1)

# 2. Training Participation Tier
df_cleaned['training_tier'] = pd.cut(
    df_cleaned['training_hours'],
    bins=[-0.1, 15.0, 35.0, 100.0],
    labels=['Minimal (<15h)', 'Moderate (15-35h)', 'High (>35h)']
)

# 3. Performance Classification
df_cleaned['performance_category'] = pd.cut(
    df_cleaned['performance_rating'],
    bins=[0.0, 2.4, 3.4, 4.4, 5.0],
    labels=['Needs Improvement', 'Meets Expectations', 'Exceeds Expectations', 'Outstanding']
)

# 4. Cost Efficiency
df_cleaned['cost_per_training_hour'] = (df_cleaned['hr_cost'] / np.maximum(df_cleaned['training_hours'], 1.0)).round(2)

# 5. Flight Risk Anomaly Flag
expected_salary = 450000 + (df_cleaned['experience_years'] * 150000)
df_cleaned['is_flight_risk'] = (
    (df_cleaned['training_hours'] >= 40.0) & 
    (df_cleaned['performance_rating'] >= 4.0) & 
    (df_cleaned['salary'] < expected_salary)
)

print("Engineered features summary:")
df_cleaned[['rating_improvement', 'rating_uplift_pct', 'training_tier', 'cost_per_training_hour', 'is_flight_risk']].head()


Engineered features summary:


,rating_improvement,rating_uplift_pct,training_tier,cost_per_training_hour,is_flight_risk
0,0.7,21.9,High (>35h),1020.95,False
1,-0.4,-13.3,Minimal (<15h),1472.03,False
2,-0.3,-9.4,Minimal (<15h),1174.84,False
3,0.8,23.5,High (>35h),1235.54,False
4,1.1,36.7,Moderate (15-35h),1281.89,False


### 8. Export Clean Dataset

In [11]:
clean_path = os.path.join("..", "data", "clean", "training_effectiveness_cleaned.csv")
if not os.path.exists(os.path.dirname(clean_path)):
    clean_path = os.path.join("data", "clean", "training_effectiveness_cleaned.csv")

# Format date columns back to clean ISO string
df_cleaned['join_date'] = df_cleaned['join_date'].dt.strftime('%Y-%m-%d')
df_cleaned['exit_date'] = df_cleaned['exit_date'].dt.strftime('%Y-%m-%d')

df_cleaned.to_csv(clean_path, index=False)
print(f"Data successfully cleaned and exported to {clean_path}")
print(f"Final shape: {df_cleaned.shape}")


Data successfully cleaned and exported to data\clean\training_effectiveness_cleaned.csv
Final shape: (5000, 28)
